# Installation and tiny synthetic smoke test
Create `environment.yml`, activate `raftup_revision`, then install `python -m pip install -e '.[revision]'` from the repository root. Register the kernel as described in README.md.
This tiny example checks the actual solver; it is not a biological benchmark or paper result.


## Data and local paths
Data are not uploaded to GitHub. Download sources and preparation instructions: [DATA.md](DATA.md).
Default prepared-input folder: `data/revision/smoke/` in the repository; set `RAFTUP_TUTORIAL_DATA` to override it. Relative paths inside `pair.json` resolve against that file. Outputs go to `outputs/revision/00_installation_smoke_test/`; set `RAFTUP_OUTPUT_ROOT` to override. The real-data notebooks do not substitute synthetic data when inputs are missing.


In [ ]:
from pathlib import Path
import os, json
import numpy as np
import raftup
from raftup import run_automated_alignment, run_manual_alignment
from raftup.experiments import load_pair, save_alignment, coverage
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), Path.cwd())
DATA = Path(os.environ.get('RAFTUP_TUTORIAL_DATA', str(REPO / 'data/revision/smoke'))).expanduser().resolve()
OUT = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', str(REPO / 'outputs/revision/00_installation_smoke_test'))).expanduser().resolve()
print('Inputs:', DATA, '\nOutputs:', OUT)
print('RAFT-UP:', raftup.__version__, raftup.__file__)


In [ ]:
import anndata as ad
from scipy.spatial.distance import cdist
xy = np.array([[0.,0.],[10.,0.],[0.,10.],[10.,10.]])
a = ad.AnnData(np.eye(4)); b = ad.AnnData(np.eye(4))
a.obsm['spatial'] = xy.copy(); b.obsm['spatial'] = xy.copy()
cost = cdist(np.eye(4), np.eye(4))
P, decision = run_manual_alignment(a, b, cost, alignment_mode='window',
    downsampling_ratio=1., feature_cutoff_downsample=.1,
    spatial_cutoff_downsample_um=1., technology='dlpfc', pixels_per_100um=100,
    random_state=42)
assert np.array_equal(P > 0, np.eye(4, dtype=bool))
print(coverage(P))
